# 03 — Analyse statistique

**Données** : `train.parquet` (80 000 clients). `Customer_ID` et `ethnic` sont exclues.

**Tests** :
- **numériques** : Mann-Whitney U + corrélation rank-biserial (taille d'effet d'un décalage monotone) ;
- **catégorielles et binaires** (dont les indicateurs `_was_negative` et les indicateurs de manquants par motif, D22) : χ² d'indépendance (Fisher exact si effectifs attendus < 5) + V de Cramér ;
- **relations non monotones** : écart maximal de taux de churn entre déciles (ou modalités) et information mutuelle ;
- **correction de Holm** sur l'ensemble des tests, comme en E2 et E4.

Seuils de taille d'effet (Cohen, valables pour |r| et pour V car la cible est binaire) : < 0,1 négligeable, 0,1-0,3 faible, 0,3-0,5 moyen, > 0,5 fort.

Logique dans `churn.evaluation.stats`.

In [1]:
import numpy as np
import pandas as pd
import plotly.io as pio
from IPython.display import display

from churn import eda
from churn.charts import (dendrogram_chart, effect_vs_spread_chart, importance_chart, publish)
from churn.config import get_config
from churn.data.audit import missing_indicators
from churn.data.split import load_train
from churn.evaluation import stats as st
from churn.logging_setup import setup_logging

setup_logging()
pio.renderers.default = "plotly_mimetype"
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 130)

cfg = get_config()
TARGET = cfg.data.target
train = load_train()

indicators = missing_indicators(train, exclude=("ethnic",))
BINARY_NUMERIC = ["truck", "rv", "forgntvl"]
NUMERIC = [c for cols in eda.VARIABLE_BLOCKS.values() for c in cols if c not in BINARY_NUMERIC]
FLAGS = [c for c in train.columns if c.endswith("_was_negative")]
CATEGORICAL = [c for c in train.select_dtypes("category").columns if c != "ethnic"]
BINARY = BINARY_NUMERIC + FLAGS + list(indicators.columns)

data = pd.concat([train, indicators], axis=1)
data["crclscod"] = eda.credit_class(train["crclscod"])   # 53 codes -> classes principales (E4)
print(f"{len(NUMERIC)} numériques, {len(CATEGORICAL)} catégorielles, {len(BINARY)} binaires "
      f"(dont {len(FLAGS)} indicateurs de négatifs et {len(indicators.columns)} indicateurs de manquants)")

74 numériques, 20 catégorielles, 24 binaires (dont 4 indicateurs de négatifs et 17 indicateurs de manquants)


## 1. Pourquoi les p-values seules ne suffisent pas avec n = 80 000

La p-value répond à la question « cet écart pourrait-il être dû au hasard ? », pas à « cet écart est-il important ? ». Pour un effet donné, elle **diminue mécaniquement quand n augmente** : l'erreur-type d'une différence décroît en $1/\sqrt{n}$. Avec 80 000 clients, un écart minuscule, sans aucun intérêt pratique, devient « significatif ».

Démonstration : la même variable (`change_mou`), sur des sous-échantillons de taille croissante.

In [2]:
st.p_value_vs_sample_size(train["change_mou"], train[TARGET], [500, 2_000, 10_000, 40_000, 80_000])

,p_value,rank_biserial
n,,
500,3.211360e-02,-0.112271
2000,7.731591e-03,-0.069189
10000,6.825693e-10,-0.071572
40000,5.330026e-29,-0.064809
80000,1.992186e-49,-0.060615


**Lecture** : la p-value passe de 0,03 à 10⁻⁴⁹ alors que l'effet se stabilise autour de −0,06 (négligeable). Conclusion : on **filtre** avec la p-value (corrigée pour la multiplicité des tests) et on **décide** avec la taille d'effet.

## 2. Pourquoi une corrélation, même de rang, peut manquer une variable importante

Une corrélation (Pearson, Spearman) ou la rank-biserial mesurent une tendance **monotone** : « plus x est grand, plus (ou moins) on churne ». `months` a un effet en **pic** : churn bas de 6 à 10 mois, très haut à 11-12 mois, moyen ensuite. Les deux côtés du pic se compensent, et la mesure monotone est presque nulle.

In [3]:
months_mw = st.mann_whitney_rank_biserial(train["months"], train[TARGET])
months_classes = st.chi2_cramers_v(eda.business_segments(train)["anciennete"].astype(str), train[TARGET])
spread, shape = st.class_profile(train, "months", TARGET, numeric=True)
pd.DataFrame({
    "mesure": ["rank-biserial (monotone)", "V de Cramér, 3 classes 6-10 / 11-12 / >= 13 mois",
               "écart entre déciles (points)", "forme de la courbe",
               "information mutuelle (nats)"],
    "valeur": [round(months_mw["effet"], 3), round(months_classes["effet"], 3), round(spread, 1),
               shape, round(st.mutual_information(train, "months", TARGET, numeric=True), 4)],
}).set_index("mesure")

,valeur
mesure,
rank-biserial (monotone),0.053
"V de Cramér, 3 classes 6-10 / 11-12 / >= 13 mois",0.173
écart entre déciles (points),28.7
forme de la courbe,non monotone
information mutuelle (nats),0.0184


**Lecture** : avec une mesure monotone, `months` serait classée **négligeable** (r = 0,05). Découpée en classes, elle obtient un V de 0,17, l'un des plus forts du jeu. L'**écart entre déciles** et l'**information mutuelle** (nulle si et seulement si variable et cible sont indépendantes, quelle que soit la forme du lien) captent ces effets : ce sont eux qui servent au classement final.

## 3. Tableau des tests (Holm sur l'ensemble)

In [4]:
results = st.univariate_table(data, TARGET, NUMERIC, CATEGORICAL + BINARY)
summary = pd.DataFrame({
    "tests": [len(results)],
    "significatifs (Holm, 5 %)": [int(results["significatif_holm"].sum())],
    **{f"effet {k}": [int((results["interpretation"] == k).sum())]
       for k in ["négligeable", "faible", "moyen", "fort"]},
    "signalées non monotones": [int(results["relation_non_monotone"].sum())],
})
display(summary)
display(results["test"].value_counts().to_frame("nombre de tests"))

,tests,"significatifs (Holm, 5 %)",effet négligeable,effet faible,effet moyen,effet fort,signalées non monotones
0,118,93,116,2,0,0,25


,nombre de tests
test,
Mann-Whitney U,74
χ²,42
Fisher exact,2


**Conclusion** : 118 tests, **93 significatifs** après Holm, mais **116 tailles d'effet négligeables** et seulement 2 faibles (`eqpdays` r = +0,149, `hnd_price` r = −0,118). Aucun effet moyen ou fort. C'est exactement la situation où la p-value seule tromperait : tout « compte » statistiquement, presque rien ne compte individuellement.

Les 25 variables non significatives sont surtout socio-démographiques (`income`, `adults`, `numbcars`, `kid*`, `truck`, `rv`, `forgntvl`, `new_cell`), des variables *data* rares (`drop_dat_Mean`, `recv_sms_Mean`, `callfwdv_Mean`), les 4 indicateurs de valeurs négatives et `roam_Mean`. Deux tests utilisent Fisher exact (indicateurs de négatifs à 2 et 4 cas positifs).

### Vérification : V de Cramér identiques à E4

Les χ² et V de E4 n'avaient été calculés que dans les sorties du notebook 02, sans être enregistrés. Ils sont recalculés ici avec la même formule, pour obtenir les p-values nécessaires à la correction de Holm, puis **enregistrés dans `reports/`** pour les étapes suivantes.

In [5]:
e4 = eda.categorical_contrast(data.assign(**{TARGET: train[TARGET]}), CATEGORICAL)["cramers_v"]
diff = (results.loc[e4.index, "effet"] - e4).abs()
print(f"écart maximal avec E4 : {diff.max():.4f} ({diff.idxmax()})")

écart maximal avec E4 : 0.0005 (dwlltype)


**Conclusion** : écart maximal de 0,0005 avec les V de E4 (dû au regroupement des modalités rares, absent en E4) : les résultats de E4 sont confirmés. Ils sont désormais enregistrés dans `reports/stats_univariate.csv`.

## 4. Tableau final trié par importance (information mutuelle)

In [6]:
final_cols = ["rang", "type", "test", "statistique", "p_ajustee_holm", "effet", "interpretation",
              "ecart_classes_pts", "forme_courbe", "info_mutuelle", "relation_non_monotone"]
final = results[final_cols].copy()
final["statistique"] = final["statistique"].map(lambda v: f"{v:.4g}")
final["p_ajustee_holm"] = final["p_ajustee_holm"].map(lambda v: f"{v:.1e}")
final["effet"] = final["effet"].round(3)
final["ecart_classes_pts"] = final["ecart_classes_pts"].round(1)
final["info_mutuelle"] = final["info_mutuelle"].round(4)
final

,rang,type,test,statistique,p_ajustee_holm,effet,interpretation,ecart_classes_pts,forme_courbe,info_mutuelle,relation_non_monotone
variable,,,,,,,,,,,
months,1,numérique,Mann-Whitney U,8.426e+08,3.3e-37,0.053,négligeable,28.7,non monotone,0.0184,True
eqpdays,2,numérique,Mann-Whitney U,9.165e+08,6.3e-289,0.149,faible,23.9,non monotone,0.0165,False
totmrc_Mean,3,numérique,Mann-Whitney U,7.268e+08,2.4e-92,-0.084,négligeable,13.4,monotone,0.0118,True
hnd_price,4,numérique,Mann-Whitney U,6.93e+08,1.4e-186,-0.118,faible,20.8,monotone,0.0064,False
adjrev,5,numérique,Mann-Whitney U,8.335e+08,6.2e-23,0.042,négligeable,12.1,non monotone,0.0056,True
mouowylisv_Mean,6,numérique,Mann-Whitney U,7.495e+08,6.2e-52,-0.063,négligeable,9.2,monotone,0.0055,False
opk_vce_Mean,7,numérique,Mann-Whitney U,7.435e+08,5.5e-65,-0.071,négligeable,12.4,monotone,0.0047,True
change_rev,8,numérique,Mann-Whitney U,7.77e+08,1.7e-01,-0.011,négligeable,6.7,non monotone,0.0045,False
change_mou,9,numérique,Mann-Whitney U,7.379e+08,1.8e-47,-0.061,négligeable,10.1,monotone,0.0043,True


In [7]:
publish(importance_chart(results, "Ancienneté et âge du terminal dominent, loin devant le reste"),
        "03_importance", width=1100)

**Conclusion** : l'information mutuelle place **`months` en 1re position** (0,018 nat) alors que sa mesure monotone est négligeable, puis `eqpdays` (0,017), `totmrc_Mean` (0,012) et `hnd_price` (0,006). Au-delà, les valeurs sont toutes entre 0,002 et 0,006 nat : l'estimateur a une précision de l'ordre de 0,001 nat, **le classement au-delà du 4e rang est donc indicatif**.

Deux remarques :

- `change_rev` (8e) a une information mutuelle notable mais une rank-biserial nulle (−0,011, non significative) : sa courbe est non monotone (les fortes hausses **et** les fortes baisses de facture vont avec plus de churn) ;
- l'information mutuelle favorise les effets qui touchent **beaucoup** de clients : `manquant_change_mou_bloc` a un écart de 26,7 points mais ne concerne que 718 clients, d'où une information mutuelle faible. Écart entre classes et information mutuelle sont complémentaires.

In [8]:
publish(effect_vs_spread_chart(results, "months : effet monotone négligeable, mais le plus grand écart entre déciles", label_top=5),
        "03_effet_vs_ecart", width=1000)
display(results.loc[results["relation_non_monotone"], ["effet", "ecart_classes_pts", "forme_courbe", "info_mutuelle"]].round(3))

,effet,ecart_classes_pts,forme_courbe,info_mutuelle
variable,,,,
months,0.053,28.72,non monotone,0.018
totmrc_Mean,-0.084,13.43,monotone,0.012
adjrev,0.042,12.10,non monotone,0.006
opk_vce_Mean,-0.071,12.35,monotone,0.005
change_mou,-0.061,10.08,monotone,0.004
hnd_webcap,0.092,11.90,sans objet,0.004
mou_cvce_Mean,-0.081,11.94,monotone,0.004
avg3mou,-0.067,12.64,monotone,0.004
attempt_Mean,-0.067,12.21,monotone,0.004


**Conclusion** : 25 variables ont un effet monotone négligeable mais un écart entre classes supérieur à 10 points. Elles ne sont pas toutes « non monotones » au même titre :

- **vraiment non monotones** (courbe en pic ou en creux) : `months` (28,7 pts), `adjrev` et `totrev` (12 pts ; ces totaux sur la durée de vie héritent du pic de `months`, voir le dendrogramme) ;
- **monotones mais faibles** : la plupart des variables d'usage (`mou_Mean`, `totmrc_Mean`, `opk_vce_Mean`…) : r entre −0,06 et −0,09, juste sous le seuil de 0,1, et une pente régulière qui s'accumule sur 10 déciles ;
- **catégorielles et indicateurs** : `hnd_webcap`, `crclscod`, `dualband`, `area`, et les indicateurs de manquants (`change_mou`, bloc d'usage, `hnd_price`), dont l'effet est fort sur des groupes petits.

Aucune de ces variables ne doit être écartée sur la seule base de sa taille d'effet monotone.

## 5. Redondance entre variables numériques

### 5.1 Paires de Spearman > 0,9

In [9]:
corr = train[NUMERIC].corr(method="spearman")
pairs = st.spearman_redundant_pairs(corr, threshold=0.9)
print(f"{len(pairs)} paires avec |ρ| > 0,9")
pairs

60 paires avec |ρ| > 0,9


,var_1,var_2,rho
0,totcalls,adjqty,0.999
1,totmou,adjmou,0.999
2,plcd_vce_Mean,attempt_Mean,0.999
3,peak_dat_Mean,mou_pead_Mean,0.999
4,comp_vce_Mean,complete_Mean,0.999
5,comp_dat_Mean,mou_cdat_Mean,0.999
6,opk_dat_Mean,mou_opkd_Mean,0.999
7,ccrndmou_Mean,cc_mou_Mean,0.998
8,ovrrev_Mean,ovrmou_Mean,0.998
9,totrev,adjrev,0.996


### 5.2 Clustering hiérarchique des variables

Distance $1 - |\rho|$, lien moyen. La ligne pointillée ($|\rho| = 0{,}9$) coupe l'arbre : les variables reliées à gauche de cette ligne forment un groupe redondant.

In [10]:
publish(dendrogram_chart(corr, "Les variables d'usage forment de grands groupes quasi interchangeables"),
        "03_dendrogramme", width=1000)

In [11]:
groups = st.variable_clusters(corr, threshold=0.9)
representatives = st.choose_representatives(groups, results["info_mutuelle"],
                                            preferred=st.PREFERRED_REPRESENTATIVES)
view = representatives.sort_values(["groupe", "garder"], ascending=[True, False])
print(f"{view['groupe'].nunique()} groupes redondants, {int((~view['garder']).sum())} variables à retirer")
view.round(4)

17 groupes redondants, 30 variables à retirer


,groupe,representante,garder,info_mutuelle
variable,,,,
peak_dat_Mean,3,peak_dat_Mean,True,0.0000
mou_pead_Mean,3,peak_dat_Mean,False,0.0000
opk_dat_Mean,4,opk_dat_Mean,True,0.0000
mou_opkd_Mean,4,opk_dat_Mean,False,0.0000
comp_dat_Mean,5,comp_dat_Mean,True,0.0003
plcd_dat_Mean,5,comp_dat_Mean,False,0.0000
mou_cdat_Mean,5,comp_dat_Mean,False,0.0001
totrev,10,totrev,True,0.0029
adjrev,10,totrev,False,0.0056


**Conclusion** : 60 paires dépassent |ρ| = 0,9 ; le clustering les regroupe en **17 groupes**, soit **30 variables à retirer** pour une régression logistique. Les groupes sont logiques :

- mêmes appels comptés de plusieurs façons (tentés, passés, aboutis, en heures pleines : `attempt_Mean`, `plcd_vce_Mean`, `comp_vce_Mean`, `complete_Mean`, `peak_vce_Mean`) ;
- moyennes sur 3 mois, 6 mois et depuis l'ouverture (`mou_Mean`, `avg3mou`, `avg6mou`, `avg3qty`, `avg6qty`) ;
- quantité et durée des mêmes événements (`ovrrev_Mean` / `ovrmou_Mean` / `vceovr_Mean`, `custcare_Mean` / `ccrndmou_Mean` / `cc_mou_Mean`).

**Choix des représentantes** : à l'intérieur d'un groupe, les écarts d'information mutuelle sont inférieurs à 0,001 nat (bruit de l'estimateur) : ils ne départagent pas. On garde la variable **la plus interprétable et déjà utilisée** en EDA et en segmentation (`mou_Mean`, `rev_Mean`, `custcare_Mean`, `ovrrev_Mean`…), sinon la plus liée au churn.

### 5.3 VIF sur les variables restantes (itération jusqu'à VIF < 10)

Variables préparées comme pour une régression logistique : imputation médiane, log si très asymétrique, standardisation. VIF$_j = 1 / (1 - R^2_j)$ : à quel point la variable $j$ est prédictible par les autres.

In [12]:
remaining = [c for c in NUMERIC if c not in representatives.index[~representatives["garder"]]]
X = st.prepare_for_linear(train, remaining)
print(f"{len(remaining)} variables après redondance ; VIF les plus élevés avant itération :")
display(st.vif_table(X).head(10).round(1).to_frame())
kept, vif_removed = st.iterative_vif(X, threshold=10)
print(f"retirées par le VIF : {len(vif_removed)}")
display(vif_removed)
print(f"{len(kept)} variables conservées ; VIF maximal final : {st.vif_table(X[kept]).iloc[0]:.2f}")
st.vif_table(X[kept]).head(8).round(2).to_frame()

44 variables après redondance ; VIF les plus élevés avant itération :


,vif
totcalls,134.2
avgqty,124.4
totrev,59.5
comp_dat_Mean,47.8
avgrev,38.5
drop_blk_Mean,38.0
attempt_Mean,36.3
opk_dat_Mean,18.3
mou_cvce_Mean,18.1
drop_vce_Mean,16.7


retirées par le VIF : 6


,vif,etape
variable,,
totcalls,134.19,1
comp_dat_Mean,47.76,2
drop_blk_Mean,37.93,3
attempt_Mean,33.89,4
opk_vce_Mean,11.93,5
totrev,11.85,6


38 variables conservées ; VIF maximal final : 6.72


,vif
rev_Mean,6.72
recv_vce_Mean,6.09
mou_cvce_Mean,5.99
mouowylisv_Mean,5.15
avgrev,5.02
avgqty,4.95
unan_vce_Mean,4.66
mou_Mean,4.49


**Conclusion** : sur les 44 variables restantes, le VIF atteint encore 134 (`totcalls`). Six itérations retirent `totcalls`, `comp_dat_Mean`, `drop_blk_Mean`, `attempt_Mean`, `opk_vce_Mean` et `totrev`. Il reste **38 variables, VIF maximal 6,7**.

Pourquoi ces variables sont colinéaires sans être redondantes deux à deux :

- **totaux sur la durée de vie** : `totcalls` ≈ moyenne mensuelle × ancienneté ; en log, log(`totcalls`) ≈ log(`avgqty`) + log(`months`), donc prédictible par ces deux variables (même chose pour `totrev` avec `avgrev`). Aucune information perdue ;
- **sommes exactes** : `drop_blk_Mean` = appels coupés + bloqués (règle vérifiée en E2), donc prédictible par ses composantes ;
- `comp_dat_Mean`, `attempt_Mean`, `opk_vce_Mean` : combinaisons des autres mesures d'appels.

Ces retraits ne concernent que la **régression logistique** (coefficients instables en cas de colinéarité) ; les modèles à arbres peuvent garder toutes les variables.

## 6. Sorties pour la suite

In [13]:
paths = st.export_outputs(results, representatives, vif_removed, cfg.paths.reports_dir)
for kind, path in paths.items():
    print(f"{kind:10} -> {path}")
top15 = pd.read_csv(paths["top"], index_col=0)
top15

univariate -> C:\Users\malek\OneDrive\Desktop\telecom_churn\reports\stats_univariate.csv
top        -> C:\Users\malek\OneDrive\Desktop\telecom_churn\reports\top15_importance.csv
drop       -> C:\Users\malek\OneDrive\Desktop\telecom_churn\reports\logreg_drop_list.json


,rang,type,test,effet,interpretation,ecart_classes_pts,forme_courbe,info_mutuelle,p_ajustee_holm,a_retirer_logreg,representante
variable,,,,,,,,,,,
months,1,numérique,Mann-Whitney U,0.053380,négligeable,28.72,non monotone,0.018357,3.308452e-37,False,months
eqpdays,2,numérique,Mann-Whitney U,0.148948,faible,23.86,non monotone,0.016550,6.254825e-289,False,eqpdays
totmrc_Mean,3,numérique,Mann-Whitney U,-0.084169,négligeable,13.43,monotone,0.011849,2.367248e-92,False,totmrc_Mean
hnd_price,4,numérique,Mann-Whitney U,-0.118477,faible,20.84,monotone,0.006430,1.370903e-186,False,hnd_price
adjrev,5,numérique,Mann-Whitney U,0.041996,négligeable,12.10,non monotone,0.005574,6.169294e-23,True,"totrev (retirée, VIF)"
mouowylisv_Mean,6,numérique,Mann-Whitney U,-0.063044,négligeable,9.22,monotone,0.005527,6.229356e-52,False,mouowylisv_Mean
opk_vce_Mean,7,numérique,Mann-Whitney U,-0.070575,négligeable,12.35,monotone,0.004718,5.455814e-65,True,"opk_vce_Mean (retirée, VIF)"
change_rev,8,numérique,Mann-Whitney U,-0.010922,négligeable,6.68,non monotone,0.004535,1.695406e-01,False,change_rev
change_mou,9,numérique,Mann-Whitney U,-0.060615,négligeable,10.08,monotone,0.004282,1.812889e-47,False,change_mou


## Conclusion

1. **Significatif ne veut pas dire important** : 93 tests sur 118 sont significatifs après Holm, mais 116 effets sont négligeables. Le signal de chaque variable est faible ; le modèle devra combiner beaucoup de petits effets.
2. **Une mesure monotone ne suffit pas** : `months`, 1re variable par information mutuelle et par écart entre déciles (28,7 pts), serait classée négligeable par sa rank-biserial (0,05). Le feature engineering (E6) devra exprimer ses effets en marche d'escalier (indicateur 11-12 mois, tranches), et les modèles à arbres les captent naturellement.
3. **Redondance massive** : 60 paires au-delà de 0,9, 17 groupes, 30 variables redondantes ; avec le VIF, **36 variables numériques à retirer** pour la régression logistique, 38 conservées (VIF max 6,7).
4. **Sorties** : `reports/stats_univariate.csv` (tableau complet), `reports/top15_importance.csv` (guide pour E6, avec la représentante de chaque variable redondante), `reports/logreg_drop_list.json` (liste utilisée en E7).